# Phase 1: Data Ingestion

**Notebook:** `01_load_documents.ipynb`  
**Matching module:** `03_ingestion/01_load_documents.py`

**Purpose:** Load PDFs, CSV files, FAQs, hospital policy files, doctor schedules, department information, website pages, database tables, manuals, and de-identified support logs into one consistent document schema.

## Input and output contract

**Input root:** `01_data/raw/`, scanned recursively.

**Data outputs:** `01_loaded_documents.json`, `01_ingestion_manifest.json`, `01_source_inventory.csv`, and `01_failed_documents.json`.

**Plot outputs:** `plots/01_documents_by_source_type.png` and `plots/01_files_by_source_category.png`.

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import sys
from collections import Counter
from pathlib import Path

from IPython.display import Image, display


def find_project_root(start: Path) -> Path:
    """Locate the project from Jupyter or repository execution."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "03_ingestion" / "01_load_documents.py").is_file():
            return candidate
        nested = candidate / "hospital_patient_helpdesk_chatbot"
        if (nested / "03_ingestion" / "01_load_documents.py").is_file():
            return nested
    raise FileNotFoundError("Could not locate hospital_patient_helpdesk_chatbot.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "01_data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "01_data" / "processed"
MODULE_PATH = PROJECT_ROOT / "03_ingestion" / "01_load_documents.py"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DATA_DIR}")
print(f"Processed data: {PROCESSED_DATA_DIR}")


In [ ]:
def load_module(module_path: Path):
    """Load the numbered workflow module from its file path."""
    specification = importlib.util.spec_from_file_location("load_documents_module", module_path)
    if specification is None or specification.loader is None:
        raise ImportError(f"Could not load module: {module_path}")
    module = importlib.util.module_from_spec(specification)
    sys.modules[specification.name] = module
    specification.loader.exec_module(module)
    return module


ingestion = load_module(MODULE_PATH)
print(f"Supported extensions: {', '.join(ingestion.SUPPORTED_EXTENSIONS)}")


## Loader strategy

- PDFs are extracted with `pypdf`, falling back to the local `pdftotext` command.
- CSV, JSON, JSONL, and SQLite sources create one normalized document per record.
- HTML pages contribute visible text while script and style content is ignored.
- SQL and text files contribute one document per file.
- Unsupported files are recorded as skipped; supported-file errors are recorded without aborting the full ingestion run.

In [ ]:
source_files = ingestion.discover_source_files(RAW_DATA_DIR)
category_counts = Counter(path.relative_to(RAW_DATA_DIR).parts[0] for path in source_files)
extension_counts = Counter(path.suffix.lower() for path in source_files)

print(f"Files discovered: {len(source_files)}")
print("Files by raw-data category:")
for category, count in sorted(category_counts.items()):
    print(f"- {category}: {count}")
print("\nFiles by extension:")
for extension, count in sorted(extension_counts.items()):
    print(f"- {extension}: {count}")


In [ ]:
preview_extensions = [".pdf", ".csv", ".json", ".html", ".db"]
for extension in preview_extensions:
    path = next((item for item in source_files if item.suffix.lower() == extension), None)
    if path is None:
        continue
    records = ingestion.LOADERS[extension](RAW_DATA_DIR, path)
    first = records[0]
    print(f"\n{path.relative_to(RAW_DATA_DIR)} -> {len(records)} record(s)")
    print(f"ID: {first.document_id}")
    print(first.text[:240])


In [ ]:
result = ingestion.run_ingestion(
    raw_data_dir=RAW_DATA_DIR,
    processed_data_dir=PROCESSED_DATA_DIR,
)
ingestion.print_result(result)


In [ ]:
documents = json.loads(result.documents_path.read_text(encoding="utf-8"))
manifest = json.loads(result.manifest_path.read_text(encoding="utf-8"))
failures = json.loads(result.failures_path.read_text(encoding="utf-8"))
with result.inventory_path.open(encoding="utf-8", newline="") as handle:
    inventory = list(csv.DictReader(handle))

if len(documents) != manifest["documents_created"]:
    raise RuntimeError("Document count does not match the manifest.")
if len({document["document_id"] for document in documents}) != len(documents):
    raise RuntimeError("Duplicate normalized document IDs found.")
if len(inventory) != manifest["files_discovered"]:
    raise RuntimeError("Inventory count does not match discovered files.")

print("Ingestion validation passed.")
print(json.dumps(manifest, indent=2))
print(f"Failure records: {len(failures)}")


## Diagnostic plots

The source-type plot shows how many normalized records each format contributes. The source-category plot shows the number of physical input files in each raw-data folder. Together they separate record volume from file volume.

In [ ]:
display(Image(filename=str(result.source_type_plot_path)))
display(Image(filename=str(result.source_category_plot_path)))


In [ ]:
print("Output files:")
for output_name in manifest["output_files"]:
    output_path = PROCESSED_DATA_DIR / output_name
    print(f"- {output_path}: {output_path.stat().st_size:,} bytes")


## Notebook and Python module roles

The notebook imports the shared module and adds guided source inspection, loader previews, manifest review, and inline plot display. The Python module contains reusable schemas, loaders, validation, reporting, plotting, output writing, and command-line behavior for automation.

## Safety and next step

- The bundled corpus is synthetic and contains no real patient records.
- Unsupported files are not silently treated as hospital knowledge.
- Ingestion extracts and labels source content without diagnosis or clinical interpretation.
- Continue with `02_clean_documents.ipynb` using `01_loaded_documents.json`.